In [ ]:
import sys
sys.path.append("..")          

from pathlib import Path
import pandas as pd
from src.data import load_ham10000
from src.train import get_device
from src.experiment import CONFIGS, run_experiment

# Medium run on a CPU: fewer images, fewer epochs, fewer seeds than the full experiment
SETTINGS = dict(epochs=3, n_train=2000, n_eval=800)   
SEEDS = [0, 1, 2]                                     
TAG = "medium_"                                      

RESULTS_FILE = Path(f"../results/{TAG}internal_results.csv")

device = get_device()                                
print("Using:", device)

df = load_ham10000("../data/ham10000", "../results/ham10000_split.csv")

Using: cpu


In [ ]:
done = set()
if RESULTS_FILE.exists():
    old = pd.read_csv(RESULTS_FILE)
    done = set(zip(old["config"], old["seed"]))

# Go through every seed and every configuration (A, B, C, D)
for seed in SEEDS:
    for name in CONFIGS:
        if (name, seed) in done:
            print("Skipping", name, seed, "(already done)")   
            continue
        print("Running", name, "seed", seed)
        # Train this configuration and get its scores on the test set
        row = run_experiment(df, name, seed, device, tag=TAG, **SETTINGS)
        # Add this run to the results file right away, so nothing is lost if it stops
        pd.DataFrame([row]).to_csv(RESULTS_FILE, mode="a",
                                   header=not RESULTS_FILE.exists(), index=False)

Skipping A_baseline 0 (already done)
Skipping B_augment 0 (already done)
Skipping C_weights 0 (already done)
Skipping D_both 0 (already done)
Skipping A_baseline 1 (already done)
Skipping B_augment 1 (already done)
Skipping C_weights 1 (already done)
Skipping D_both 1 (already done)
Skipping A_baseline 2 (already done)
Skipping B_augment 2 (already done)
Skipping C_weights 2 (already done)
Skipping D_both 2 (already done)


In [4]:
res = pd.read_csv(RESULTS_FILE)
cols = ["test_auc", "test_balanced_acc", "test_sensitivity", "test_specificity"]

# Average and spread (std) of each score, one row per configuration
res.groupby("config")[cols].agg(["mean", "std"]).round(3)

test_auc        test_balanced_acc        test_sensitivity         \
               mean    std              mean    std             mean    std   
config                                                                        
A_baseline    0.849  0.023             0.767  0.021            0.676  0.058   
B_augment     0.877  0.028             0.785  0.032            0.813  0.041   
C_weights     0.844  0.025             0.762  0.010            0.681  0.038   
D_both        0.882  0.019             0.790  0.035            0.841  0.042   

           test_specificity         
                       mean    std  
config                              
A_baseline            0.858  0.033  
B_augment             0.757  0.058  
C_weights             0.842  0.022  
D_both                0.738  0.054